In [43]:
from bs4 import BeautifulSoup, Tag
from requests import get
import unicodedata

In [44]:
def clean_tag_text(tag: Tag) -> str:
    text = tag.get_text(separator=" ", strip=True).lower()
    text = unicodedata.normalize("NFKC", text)
    return text

In [45]:
def get_sections(expositions):
    sections = []
    current_section = []

    for elem in expositions.children:
        if not isinstance(elem, Tag):
            continue

        if elem.name == "div" and "crp_related" in (elem.get("class") or []):
            break

        if elem.name == "h2":
            if current_section:
                sections.append(current_section)
            current_section = [elem]
        else:
            if current_section:
                current_section.append(elem)

    if current_section:
        sections.append(current_section)

    return sections

In [48]:
def get_data(sections):
    data = {"expositions": []}

    for k, section in enumerate(sections):
        if k+1 == len(sections):
            title = section[0].text
            description = section[-3].text
            cleaned_text = clean_tag_text(section[-2])
            dates, lieu = cleaned_text.split('du')[-1].strip().split('lieu : ', 1)
        else:
            title = section[0].text
            description = section[-2].text
            cleaned_text = clean_tag_text(section[-1])
            dates, lieu = cleaned_text.split('du')[-1].strip().split('lieu : ', 1)

        row = {
            'title':title,
            'description':description,
            'dates': dates,
            'lieu': lieu
        }

        data['expositions'].append(row)

    return data

In [ ]:
text = "ghost in the shell exhibition du 12 avril au 17 août 2025 lieu : setagaya literary museum lieu : site officiel"
text.split('du')[-1]

' 12 avril au 17 août 2025 lieu : setagaya literary museum lieu : site officiel'

: 

In [47]:
months = ['janvier', 'fevrier', 'mars', 'avril', 'mai', 'aout', 'octobre', 'decembre']
datas = []

for month in months:
    url = f"https://ichiban-japan.com/expositions-tokyo-{month}-2025/"
    header = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
    res = get(url, headers=header)
    soup = BeautifulSoup(res.content, 'html.parser')

    expositions = soup.find_all('div', class_='entry-content')[0]
    sections = get_sections(expositions)
    data = get_data(sections)
    datas.append(data)


ryuichi sakamoto | seeing sound, hearing time du 21 décembre 2024 au 30 mars 2025 lieu : musée d’art moderne de tokyo site officiel
ryuichi sakamoto | seeing sound, hearing time du 21 décembre 2024 au 30 mars 2025 lieu : musée d’art moderne de tokyo site officiel
feminism and the moving image du 11 février au 15 juin 2025 lieu : the national museum of modern art site officiel
joan miró du 1er mars au 6 juillet 2025 lieu : musée d’art métropolitain de tokyo site officiel
ghost in the shell exhibition du 12 avril au 17 août 2025 lieu : setagaya literary museum lieu : site officiel
mam project 033 : christine sun kim du 2 juillet au 9 novembre 2025 lieu : mori art museum site officiel
picasso ceramics: art of mitate du 29 octobre 2024 au 28 décembre 2025 lieu : yoku moku museum site officiel
taro’s space du 6 novembre 2025 au 8 mars 2026 lieu : musée okamoto taro kinenkan site officiel


In [51]:
datas[1]

{'expositions': [{'title': 'TOYOHARA KUNICHIKA (1er FÉVRIER-26 MARS 2025)',
   'description': '\n静嘉堂文庫美術館（静嘉堂＠丸の内）では「豊原国周生誕190年\u3000#歌舞伎を描く―秘蔵の浮世絵初公開」を開催中。続いて太田記念美術館でも2/1より「生誕190年\u3000豊原国周」展がいよいよオープン。豊原国周を取り上げた両展覧会、相互割引も予定しておりますのでお楽しみに。https://t.co/E0GIIi0dnT pic.twitter.com/Ag91xKftkp— 太田記念美術館 Ota Memorial Museum of Art (@ukiyoeota) January 26, 2025\n',
   'dates': '1 er février au 26 mars 2025 ',
   'lieu': 'ota memorial museum of art site officiel'},
  {'title': 'HOKUSAI: ANOTHER STORY IN TOKYO (1er FÉVRIER-1er JUIN 2025)',
   'description': 'On entre dans un voyage sensoriel au cœur des œuvres de Katsushika Hokusai, où l’image, le son et la sensation se rencontrent. L’exposition propose de plonger dans l’univers de l’artiste à travers des technologies immersives et des collaborations exclusives, qui vous permettent de ressentir la beauté de ses célèbres estampes tout en explorant l’ambiance du Japon d’Edo.',
   'dates': '1 er février au 1er juin 2025 ',
   'lieu': 'tokyu plaza sh